In [ ]:
!git clone https://github.com/Salvatorevil03/ControlNet2026.git

In [ ]:
%cd /content/ControlNet2026

In [ ]:
%%capture
!pip install -q gradio pytorch-lightning omegaconf einops test-tube streamlit streamlit-drawable-canvas webdataset kornia open_clip_torch invisible-watermark torchmetrics timm addict yapf basicsr imageio-ffmpeg opencv-contrib-python

In [ ]:
# @title ⬇️ Selezione Modelli ControlNet 1.5
# @markdown Spunta i modelli che desideri scaricare (attenzione: ogni modello pesa ~5.7 GB).

import os

# --- INIZIO FORM COLAB ---
Canny = True # @param {type:"boolean"}
Depth = True # @param {type:"boolean"}
HED = True # @param {type:"boolean"}
MLSD = True # @param {type:"boolean"}
Normal = True # @param {type:"boolean"}
OpenPose = True # @param {type:"boolean"}
Scribble = True # @param {type:"boolean"}
Segmentation = True # @param {type:"boolean"}
# --- FINE FORM COLAB ---

# Creiamo la lista dei modelli da scaricare in base alle tue spunte
models_to_download = []
if Canny: models_to_download.append("control_sd15_canny.pth")
if Depth: models_to_download.append("control_sd15_depth.pth")
if HED: models_to_download.append("control_sd15_hed.pth")
if MLSD: models_to_download.append("control_sd15_mlsd.pth")
if Normal: models_to_download.append("control_sd15_normal.pth")
if OpenPose: models_to_download.append("control_sd15_openpose.pth")
if Scribble: models_to_download.append("control_sd15_scribble.pth")
if Segmentation: models_to_download.append("control_sd15_seg.pth")

base_url = "https://huggingface.co/lllyasviel/ControlNet/resolve/main/models/"
model_dir = "/content/ControlNet2026/models"
 #/kaggle/working/ControlNet2026/models if running on Kaggle
# Assicuriamoci che la cartella esista
os.makedirs(model_dir, exist_ok=True)

if not models_to_download:
    print("⚠️ Nessun modello selezionato. Spunta almeno una casella.")
else:
    print(f"🚀 Inizio controllo e download di {len(models_to_download)} modelli...\n")

    for model in models_to_download:
        model_path = os.path.join(model_dir, model)
        if not os.path.exists(model_path):
            print(f"⬇️ Scaricando {model}...")
            # Scarica in modo silenzioso mostrando solo la progress bar
            !wget -q --show-progress -O "{model_path}" "{base_url}{model}"
        else:
            print(f"✅ {model} già presente, salto il download.")

    print("\n🎉 Download completato!")

In [ ]:
import os
import sys

"""
--- PATCH BASICSR vs TORCHVISION (V2) ---
Aggiorna l'import senza tentare di caricare la libreria rotta!
"""

print("Cerco il file difettoso di basicsr...")

# Troviamo la cartella dist-packages senza importare basicsr
dist_packages = [p for p in sys.path if 'dist-packages' in p][0]
degradations_file = os.path.join(dist_packages, 'basicsr', 'data', 'degradations.py')
print(degradations_file)

if os.path.exists(degradations_file):
    with open(degradations_file, 'r', encoding='utf-8') as f:
        content = f.read()

    vecchio_import = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    nuovo_import = "from torchvision.transforms.functional import rgb_to_grayscale"

    if vecchio_import in content:
        content = content.replace(vecchio_import, nuovo_import)

        with open(degradations_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print("✅ Patch applicata con successo! Libreria basicsr svecchiata.")
    else:
        print("⚠️ Patch già applicata o stringa non trovata.")
else:
    print(f"❌ Impossibile trovare il file: {degradations_file}")

In [ ]:
%%capture
!pip install transformers

In [ ]:
!python gradio_canny2image.py